In [42]:
import numpy as np
import pandas as pd
from google.colab import files
import io

df = pd.read_csv('Pokemon_Complete_Gen1_to_Gen9.csv')
print(df.head())

   id        name  height  weight  base_experience  type1   type2  hp  attack  \
0   1   bulbasaur       7      69               64  grass  poison  45      49   
1   2     ivysaur      10     130              142  grass  poison  60      62   
2   3    venusaur      20    1000              236  grass  poison  80      82   
3   4  charmander       6      85               62   fire     NaN  39      52   
4   5  charmeleon      11     190              142   fire     NaN  58      64   

   defense  sp_attack  sp_defense  speed              abilities  
0       49         65          65     45  overgrow, chlorophyll  
1       63         80          80     60  overgrow, chlorophyll  
2       83        100         100     80  overgrow, chlorophyll  
3       43         60          50     65     blaze, solar-power  
4       58         80          65     80     blaze, solar-power  


In [43]:
features = [
    "height",
    "weight",
    "hp",
    "attack",
    "defense",
    "sp_attack",
    "sp_defense",
    "speed"
]

df["dual_type"] = df["type2"].notna().astype(int)

X = df[features].values
Y = df["dual_type"].values.reshape(1, -1)

# Normalize input data
X = (X - X.mean(axis=0)) / X.std(axis=0)
X = X.T

In [44]:
# Split train and test data
np.random.seed(42)
m = X.shape[1]
indices = np.random.permutation(m)

train_size = int(0.8 * m)
train_idx = indices[:train_size]
test_idx = indices[train_size:]

X_train = X[:, train_idx]
Y_train = Y[:, train_idx]

X_test = X[:, test_idx]
Y_test = Y[:, test_idx]

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)
print("X_test shape:", X_test.shape)
print("Y_test shape:", Y_test.shape)

X_train shape: (8, 820)
Y_train shape: (1, 820)
X_test shape: (8, 205)
Y_test shape: (1, 205)


In [45]:
def initialize_parameters(input_size, hidden_size, output_size):
    np.random.seed(42)

    parameters = {
        "W1": np.random.randn(hidden_size, input_size) * 0.01,
        "b1": np.zeros((hidden_size, 1)),
        "W2": np.random.randn(output_size, hidden_size) * 0.01,
        "b2": np.zeros((output_size, 1))
    }

    return parameters


def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))


def relu(Z):
    return np.maximum(0, Z)


def relu_derivative(Z):
    return (Z > 0).astype(int)


def forward_propagation(X, parameters):
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]

    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)

    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    cache = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2
    }

    return A2, cache


def compute_cost(Y, A2):
    m = Y.shape[1]

    A2 = np.clip(A2, 1e-8, 1 - 1e-8)

    cost = -np.sum(Y * np.log(A2) + (1 - Y) * np.log(1 - A2)) / m

    return cost


def backward_propagation(X, Y, parameters, cache):
    m = X.shape[1]

    W2 = parameters["W2"]

    A1 = cache["A1"]
    A2 = cache["A2"]
    Z1 = cache["Z1"]

    dZ2 = A2 - Y
    dW2 = np.dot(dZ2, A1.T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    dZ1 = np.dot(W2.T, dZ2) * relu_derivative(Z1)
    dW1 = np.dot(dZ1, X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    grads = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2
    }

    return grads


def update_parameters(parameters, grads, learning_rate):
    parameters["W1"] = parameters["W1"] - learning_rate * grads["dW1"]
    parameters["b1"] = parameters["b1"] - learning_rate * grads["db1"]
    parameters["W2"] = parameters["W2"] - learning_rate * grads["dW2"]
    parameters["b2"] = parameters["b2"] - learning_rate * grads["db2"]

    return parameters


def train(X, Y, hidden_size, epochs, learning_rate):
    input_size = X.shape[0]
    output_size = 1

    parameters = initialize_parameters(input_size, hidden_size, output_size)

    for i in range(epochs):
        A2, cache = forward_propagation(X, parameters)

        cost = compute_cost(Y, A2)

        grads = backward_propagation(X, Y, parameters, cache)

        parameters = update_parameters(parameters, grads, learning_rate)

        if i % 1000 == 0:
            print("Epoch:", i, "Cost:", cost)

    return parameters


def predict(X, parameters):
    A2, cache = forward_propagation(X, parameters)

    prediction = (A2 > 0.5).astype(int)

    return prediction, A2

In [46]:
#Train Model
parameters = train(
    X_train, Y_train, hidden_size=8, epochs=10000, learning_rate=0.05
)

Epoch: 0 Cost: 0.6931741791639958
Epoch: 1000 Cost: 0.6660530611491331
Epoch: 2000 Cost: 0.660580754292801
Epoch: 3000 Cost: 0.6576129027291512
Epoch: 4000 Cost: 0.654314900989872
Epoch: 5000 Cost: 0.6523515279209932
Epoch: 6000 Cost: 0.6505174476206484
Epoch: 7000 Cost: 0.6494879644039832
Epoch: 8000 Cost: 0.6468255701159988
Epoch: 9000 Cost: 0.6437744706579351


In [47]:
#Test Model
Y_pred, Y_prob = predict(X_test, parameters)

accuracy = np.mean(Y_pred == Y_test) * 100

print("Accuracy:", accuracy, "%")

Accuracy: 60.97560975609756 %


# Analisis
- Model ANN/JST ini melakukan klasifikasi biner.
- Model memprediksi apakah suatu Pokemon bertipe Tunggal atau Tipe Ganda.
- Tipe Tunggal berarti Pokemon hanya memiliki tipe 1.
- Tipe Ganda berarti Pokemon memiliki tipe1 dan tipe2.

- Variabel inputnya adalah height, weight, hp, attack, defense, sp_attack, sp_defense, and speed.
- Model tidak menggunakan type1 dan type2 sebagai input karena type2 digunakan untuk membuat kelas target.

Hasil Akurasi: 60.97560975609756%
